In [0]:
from pyspark.sql import functions as F
from datetime import datetime
from dateutil.relativedelta import relativedelta
from zoneinfo import ZoneInfo
TS_FORMAT = "yyyy-MM-dd HH:mm"

def parse_ts(df):
    return F.to_timestamp(F.concat_ws(" ", df["Date"], df["Time"]), TS_FORMAT)

current_ts = (datetime.now(ZoneInfo("Europe/Belgrade")).replace(tzinfo=None)
          - relativedelta(years=3))

df = (spark.read
      .option("header", "true")
      .csv("/Volumes/workspace/bronze/landing/smart_home_energy_consumption.csv")
      .withColumnsRenamed({
          "Home ID": "home_id",
          "Appliance Type": "appliance_type",
          "Energy Consumption (kWh)": "energy_consumption_kwh",
          "Outdoor Temperature (°C)": "outdoor_temperature",
          "Household Size": "household_size"}))

df = df.withColumn("_ts", parse_ts(df))

condition = F.col("_ts") <= F.lit(current_ts)

#incremental loading

if spark.catalog.tableExists("workspace.bronze.readings_full"):
    t = spark.table("workspace.bronze.readings_full")
    last_loaded = t.agg(F.max(parse_ts(t))).first()[0]
    if last_loaded is not None:
        condition = condition & (F.col("_ts") > F.lit(last_loaded))

new_rows = df.filter(condition).drop("_ts")
new_rows.write.mode("append").saveAsTable("workspace.bronze.readings_full")

print(f"added {new_rows.count()} rows up to {current_ts}")

In [0]:
from pyspark.sql import functions as F

#cleaning readings table

df = spark.sql("SELECT * from workspace.bronze.readings_full")
df = df.withColumnRenamed("home_id", "household_id")
df = df.withColumn("household_id", F.col("household_id").cast("int"))
df = df.withColumn("household_size", F.col("household_size").cast("int"))
df = df.withColumn("energy_consumption_kwh", F.col("energy_consumption_kwh").cast("double"))
df = df.withColumn("outdoor_temperature", F.col("outdoor_temperature").cast("double"))

df = df.withColumn(
    "reading_ts",
    F.to_timestamp(F.concat_ws(" ", "Date", "Time"), "yyyy-MM-dd HH:mm")
)
df = df.filter(F.col("reading_ts").isNotNull())
df = df.drop("Date", "Time", "size","household_size")

if spark.catalog.tableExists("workspace.silver.readings"):
    last_ts = spark.table("workspace.silver.readings").agg(F.max("reading_ts")).first()[0]
    if last_ts is not None:
        df = df.filter(F.col("reading_ts") >= F.lit(last_ts))

key = ["household_id", "appliance_type", "reading_ts"]
df = df.dropDuplicates(key)

if not spark.catalog.tableExists("workspace.silver.readings"):
    df.write.saveAsTable("workspace.silver.readings")
else:
    df.createOrReplaceTempView("new_readings")
    spark.sql(f"""
        MERGE INTO {"workspace.silver.readings"} r
        USING new_readings n
        ON  r.household_id   = n.household_id
        AND r.appliance_type = n.appliance_type
        AND r.reading_ts     = n.reading_ts
        WHEN NOT MATCHED THEN INSERT *
    """)

print("silver.readings rows:", spark.table("workspace.silver.readings").count())

In [0]:

import pyspark.sql.functions as F
from datetime import date, timedelta, datetime
from dateutil.relativedelta import relativedelta
from zoneinfo import ZoneInfo
import calendar

df_households = spark.sql(
    "SELECT * FROM workspace.silver.households"
)

df_readings = spark.sql(
    "SELECT * FROM workspace.silver.readings"
)

current_date = date.today() - relativedelta(years=3)
current_ts = datetime.now(ZoneInfo("Europe/Belgrade")) - relativedelta(years=3)

df_readings = (
    df_readings
    .filter(F.col("reading_ts") <= F.lit(current_ts))
    .withColumn("reading_date", F.to_date("reading_ts"))
    .withColumn("reading_day", F.dayofweek("reading_ts"))
    .withColumn("reading_hour", F.hour("reading_ts"))
    .withColumn(
        "low_tariff_kwh",
        F.when(
            (F.col("reading_day") == 1) |
            (F.col("reading_hour") >= 22) |
            (F.col("reading_hour") < 6),
            F.col("energy_consumption_kwh") 
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "high_tariff_kwh",
        F.when(
            (F.col("reading_day") != 1) &
            (F.col("reading_hour") >= 6) &
            (F.col("reading_hour") < 22),
            F.col("energy_consumption_kwh")
        ).otherwise(F.lit(0.0))
    )
)

df_readings_grouped = (
    df_readings
    .groupBy("household_id")
    .agg(
        F.sum("high_tariff_kwh")
        .alias("total_high_tariff_kwh"),
        F.sum("low_tariff_kwh")
        .alias("total_low_tariff_kwh")
    )
)

daily_energy_consumption = (
    df_readings
    .filter(F.col("reading_date") == F.lit(current_date))
    .groupBy("household_id")
    .agg(
        F.sum("high_tariff_kwh")
        .alias("daily_high_tariff_kwh"),
        F.sum("low_tariff_kwh")
        .alias("daily_low_tariff_kwh")
    )
)

week_start = current_date - timedelta(
    days=current_date.weekday()
)
week_end = week_start + timedelta(days=6)

weekly_energy_consumption = (
    df_readings
    .filter(
        (F.col("reading_date") >= F.lit(week_start)) &
        (F.col("reading_date") <= F.lit(week_end)) 
    )
    .groupBy("household_id")
    .agg(
        F.sum("high_tariff_kwh")
        .alias("weekly_high_tariff_kwh"),
        F.sum("low_tariff_kwh")
        .alias("weekly_low_tariff_kwh")
    )
)

month_start = current_date.replace(day=1)
month_end = current_date.replace(
    day=calendar.monthrange(
        current_date.year, current_date.month
    )[1]
)

monthly_energy_consumption = (
    df_readings
    .filter(
        (F.col("reading_date") >= F.lit(month_start)) &
        (F.col("reading_date") <= F.lit(month_end))
    )
    .groupBy("household_id")
    .agg(
        F.sum("high_tariff_kwh")
        .alias("monthly_high_tariff_kwh"),
        F.sum("low_tariff_kwh")
        .alias("monthly_low_tariff_kwh")
    )
)

df_readings_grouped = (
    df_readings_grouped
    .join(daily_energy_consumption, on="household_id", how="left")
    .join(weekly_energy_consumption, on="household_id", how="left")
    .join(monthly_energy_consumption, on="household_id", how="left")
)

consumption_columns = [
    c for c in df_readings_grouped.columns
    if c != "household_id"
]

df_readings_grouped = df_readings_grouped.fillna(
    0, subset=consumption_columns
)

df_readings_grouped = df_readings_grouped.withColumns({
    c: F.round(F.col(c), 2)
    for c in consumption_columns
})

df_readings_grouped = df_readings_grouped.orderBy(
    "household_id"
)

df_readings_grouped.write \
    .mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable("workspace.gold.household_reports")

In [0]:

from pyspark.sql import functions as F

df_prices = (
    spark.sql("SELECT * FROM bronze.electricity_prices")
    .withColumn("lowerLimit", F.col("lowerLimit").cast("double"))
    .withColumn("upperLimit", F.col("upperLimit").cast("double"))
    .withColumn("price", F.col("price").cast("double"))
)

low_tariff_price = (
    df_prices
    .filter(F.col("tariff") == "Low tariff")
    .select("price")
    .first()["price"]
)

high_tariff_tiers = (
    df_prices
    .filter(F.col("tariff").startswith("High Tariff"))
    .select("lowerLimit", "upperLimit", "price")
    .orderBy("lowerLimit")
    .collect()
)

high_tariff_prices = [
    row.asDict()
    for row in high_tariff_tiers
]


def high_tariff_cost(consumption):

    total_cost = F.lit(0.0)

    for tier in high_tariff_tiers:
        lower = tier["lowerLimit"] or 0.0
        upper = tier["upperLimit"]
        price = tier["price"]

        if price is None:
            continue

        if upper is not None:
            kwh_in_tier = F.greatest(
                F.lit(0.0),
                F.least(consumption, F.lit(upper)) - F.lit(lower)
            )
        else:
            # The last tier has no upper limit
            kwh_in_tier = F.greatest(
                F.lit(0.0),
                consumption - F.lit(lower)
            )

        total_cost = total_cost + kwh_in_tier * F.lit(price)

    return total_cost

df_report = spark.sql("SELECT * FROM workspace.gold.household_reports")

for period in ["daily", "weekly", "monthly"]:
    df_report = df_report.withColumn(
        f"{period}_low_tariff_cost",
        F.col(f"{period}_low_tariff_kwh") * F.lit(low_tariff_price)
    )

df_report = df_report.withColumn(
    "monthly_high_tariff_cost",
    high_tariff_cost(F.col("monthly_high_tariff_kwh"))
)

for period in ["daily", "weekly"]:
    remaining_month_kwh = F.greatest(
        F.lit(0.0),
        F.col("monthly_high_tariff_kwh")
        - F.col(f"{period}_high_tariff_kwh")
    )

    df_report = df_report.withColumn(
        f"{period}_high_tariff_cost",
        high_tariff_cost(F.col("monthly_high_tariff_kwh"))
        - high_tariff_cost(remaining_month_kwh)
    )

cost_columns = [
    column for column in df_report.columns
    if column.endswith("_cost")
]

df_report = df_report.withColumns({
    column: F.round(F.col(column), 2)
    for column in cost_columns
})

df_report = df_report.orderBy("household_id")

df_report.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.gold.household_reports")


In [0]:
from pyspark.sql import functions as F
from datetime import date,timedelta, datetime
from zoneinfo import ZoneInfo 
import calendar

df = spark.sql("SELECT * FROM workspace.silver.readings")

current_date = date.today() - relativedelta(years=3)
current_ts = datetime.now(ZoneInfo("Europe/Belgrade")) - relativedelta(years=3)

week_start = current_date - timedelta(days=current_date.weekday())
week_end = week_start + timedelta(days=6)
month_start = current_date.replace(day=1)
month_end = current_date.replace(day=calendar.monthrange(current_date.year, current_date.month)[1])

kwh = F.col("energy_consumption_kwh")

df = (
    df
    .filter(F.col("reading_ts") <= F.lit(current_ts))
    .withColumn("reading_date", F.to_date("reading_ts"))
    .withColumn("reading_day", F.dayofweek("reading_ts"))
    .withColumn("reading_hour", F.hour("reading_ts"))
    .withColumn(
        "low_tariff_kwh",
        F.when(
            (F.col("reading_day") == 1) |
            (F.col("reading_hour") >= 22) |
            (F.col("reading_hour") < 6),
            F.col("energy_consumption_kwh") 
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "high_tariff_kwh",
        F.when(
            (F.col("reading_day") != 1) &
            (F.col("reading_hour") >= 6) &
            (F.col("reading_hour") < 22),
            F.col("energy_consumption_kwh")
        ).otherwise(F.lit(0.0))
    )
)

d = F.col("reading_date")

periods = {
    "daily":   d == F.lit(current_date),
    "weekly":  d.between(F.lit(week_start), F.lit(week_end)),
    "monthly": d.between(F.lit(month_start), F.lit(month_end)),
    "total":   F.lit(True),
}

aggs = [
    F.sum(F.when(cond, F.col(f"{tariff}_tariff_kwh")).otherwise(F.lit(0.0)))
     .alias(f"{period}_{tariff}_kwh")
    for period, cond in periods.items()
    for tariff in ("low", "high")
]

appliance_consumption = (df
    .groupBy("household_id", "appliance_type")
    .agg(*aggs)
    .withColumns({c: F.round(F.col(c), 2) for c in [
        f"{period}_{tariff}_kwh" for period, cond in periods.items() for tariff in ("low", "high")]})
    .orderBy("household_id", "appliance_type")
)

appliance_consumption.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.gold.appliance_reports")